In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_percentage_error

from loaders._load_vn30_reg_deep import preprocess, VN30, TARGETS
from models.regression.tcn import TemporalConvolutionalNetwork as TCN

In [4]:
train_loader, valid_loader, test_loader, scaler = preprocess('ACB', 'tcn', verbose=True)

Train shape: torch.Size([1094, 4, 30]), torch.Size([1094, 4])
Valid shape: torch.Size([121, 4, 30]), torch.Size([121, 4])
Test shape: torch.Size([328, 4, 30]), torch.Size([328, 4])


In [ ]:
model = TCN(
    n_channels=4,
    hidden_dim=256,
    output_dim=4,
    p_dropout=0.1
)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.8)
criterion = nn.SmoothL1Loss()

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
best_val_loss = float('inf')
num_epochs = 100

for epoch in range(1, num_epochs + 1):
    # --- train ---
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_loader.dataset)

    # --- validate ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            val_loss += criterion(preds, y_batch).item() * X_batch.size(0)
    val_loss /= len(valid_loader.dataset)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}")

    # --- checkpoint ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'checkpoints_tcn/tcn_ACB.pth')
        print(f"  → Saved checkpoint | val_loss={val_loss:.6f}")

In [11]:
def eval(symbol: str):
    model = TCN(
        n_channels=4,
        hidden_dim=256,
        output_dim=4,
        p_dropout=0.1
    )
    _, _, test_loader, scaler = preprocess(symbol, 'tcn')
    model.load_state_dict(torch.load(f'checkpoints_tcn/tcn_{symbol}.pth', map_location='cpu'))
    model.eval()

    # Thu thập dự đoán và nhãn
    all_preds   = []
    all_targets = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(y_batch.numpy())

    all_preds   = np.vstack(all_preds)   # (n_samples, 5)
    all_targets = np.vstack(all_targets)

    # Inverse scaling
    all_preds_inv   = scaler.inverse_transform(all_preds)
    all_targets_inv = scaler.inverse_transform(all_targets)

    # Tính metrics
    r2   = r2_score(all_targets_inv, all_preds_inv, multioutput='uniform_average')
    mape = mean_absolute_percentage_error(all_targets_inv, all_preds_inv) * 100
    
    print(f"Symbol: {symbol}, R^2: {r2:.4f}, MAPE: {mape:.4f}")
    return r2, mape

In [12]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    r2, mape = eval(symbol)
    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R^2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R^2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R^2: 0.6408, MAPE: 2.8412
Symbol: BCM, R^2: 0.8681, MAPE: 2.6084
Symbol: BID, R^2: -0.2490, MAPE: 5.3915
Symbol: BVH, R^2: 0.9317, MAPE: 1.9427
Symbol: CTG, R^2: 0.6791, MAPE: 4.2400
Symbol: FPT, R^2: 0.9761, MAPE: 1.9295
Symbol: GAS, R^2: 0.8983, MAPE: 1.3125
Symbol: GVR, R^2: 0.8860, MAPE: 3.3570
Symbol: HDB, R^2: 0.9427, MAPE: 1.7795
Symbol: HPG, R^2: 0.7400, MAPE: 1.9098
Symbol: LPB, R^2: 0.2974, MAPE: 17.7857
Symbol: MBB, R^2: 0.7173, MAPE: 3.4032
Symbol: MSN, R^2: 0.9153, MAPE: 1.5929
Symbol: MWG, R^2: 0.9441, MAPE: 2.2351
Symbol: PLX, R^2: 0.9696, MAPE: 1.5175
Symbol: SAB, R^2: 0.0392, MAPE: 4.8730
Symbol: SHB, R^2: 0.9285, MAPE: 1.5656
Symbol: SSB, R^2: 0.8612, MAPE: 1.7161
Symbol: SSI, R^2: 0.8550, MAPE: 1.8188
Symbol: STB, R^2: 0.9490, MAPE: 1.8816
Symbol: TCB, R^2: 0.9073, MAPE: 2.5798
Symbol: TPB, R^2: 0.8883, MAPE: 1.7788
Symbol: VCB, R^2: 0.7337, MAPE: 1.3106
Symbol: VHM, R^2: 0.9186, MAPE: 1.9089
Symbol: VIB, R^2: 0.8875, MAPE: 1.3673
Symbol: VIC, R^2: 0.841